## Your pipeline, part 8: equal-weight fusion of 8 members (CPU, no GPU quota)
HW2, notebook 14 (**Kaggle CPU**, ~25-35 min).

**Why.** Notebook 12: equal weights over notebook 11's six members scored 0.2964 on the leaderboard, better than the
weights tuned on val (0.2938), although lower on val (0.2701 vs 0.2757). So nothing is tuned here: notebook 13's two new
YOLO11l models join as members 7 and 8, every member has weight 1, and the fusion settings stay those of notebook 12
(`p = 1`, IoU 0.55).

**Steps**
1. Check: the 6-member equal fusion reproduces notebook 11/12's val number (0.2701).
2. Each new model alone on val, then the 8-member equal fusion -> `submission_14_equal8.csv` (**the one to submit**).
3. For information only: 6 members without the two weakest (`04`, `08`, val ~0.205) -> `submission_14_no04_08.csv`.
   Not a recommendation: choosing members on val is the kind of fitting that notebook 12 showed does not transfer.

**Inputs:** competition; outputs of **07** (`submission_07_A2_A3_wbf.csv`), **08** (`passes_08_val.pkl.gz`,
`submission_08_A2_A3_wbf.csv`), **09** (`passes_final_04_val.pkl.gz`), **10** (`passes_10_val.pkl.gz`,
`submission_10_yolo11m.csv`), **11** (`passes_11_*_val.pkl.gz`, `submission_11_*.csv`, `results_11_val.csv`),
**13** (`passes_13_*_val.pkl.gz`, `passes_13_*_test.pkl.gz`). Accelerator **None** (CPU).

In [1]:
%pip install -q pycocotools

In [2]:
import os, io, json, time, contextlib, gc, gzip, pickle, ast
from pathlib import Path
from multiprocessing import Pool
import numpy as np, pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

WBF_IOU, MAX_DET = 0.55, 500                       # as in notebooks 07-11
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]
MERGE_WORKERS = max(1, (os.cpu_count() or 2))
VAL_STEP = 5
NEW = ["L960", "M1280", "M960b"]                   # notebook 11
NEW13 = ["L960s1", "L1088"]                         # notebook 13
VAL_PASSES = {"04": "passes_final_04_val.pkl.gz", "08": "passes_08_val.pkl.gz", "10": "passes_10_val.pkl.gz",
              **{k: f"passes_11_{k}_val.pkl.gz" for k in NEW}, **{k: f"passes_13_{k}_val.pkl.gz" for k in NEW13}}
TEST_CSVS = {"04": "submission_07_A2_A3_wbf.csv", "08": "submission_08_A2_A3_wbf.csv", "10": "submission_10_yolo11m.csv",
             **{k: f"submission_11_{k}.csv" for k in NEW}}
TEST_PASSES = {k: f"passes_13_{k}_test.pkl.gz" for k in NEW13}   # merged here (notebook 13 writes no CSVs)
missing = [f for f in [*VAL_PASSES.values(), *TEST_CSVS.values(), *TEST_PASSES.values(), "results_11_val.csv", "test_images.csv",
                       "val/annotations_coco.json"] if not find_all(f)]
assert not missing, "attach the outputs that contain: " + ", ".join(missing)
TEST_CSV = find("test_images.csv")
print(f"CPU, {os.cpu_count()} cores | all inputs found")

CPU, 4 cores | all inputs found


### Evaluator and val frames (same as notebooks 02-11)

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
VAL_GT = load_gt(find("val/annotations_coco.json"))
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_IDS = [int(i) for i in _v.id]                  # the every-5th frames of notebooks 07-11
TEST_IDS = list(pd.read_csv(TEST_CSV).image_id)
print(f"{len(VAL_IDS)} val frames | {len(TEST_IDS)} test frames")

571 val frames | 3811 test frames


### Members (merged six-pass outputs) and the fusion rule of notebooks 10-11

In [5]:
def wbf(views, iou_thr=WBF_IOU):
    # views: list of (N, 6) arrays, one per pass; returns fused (M, 6), best first, at most MAX_DET
    n_views = len(views)
    allb = np.concatenate([v for v in views if len(v)] or [np.zeros((0, 6), np.float32)])
    out = []
    for c in np.unique(allb[:, 5]):
        b = allb[allb[:, 5] == c]
        b = b[np.argsort(-b[:, 4])]
        acc = np.zeros((len(b), 4)); fb = np.zeros((len(b), 4)); w = np.zeros(len(b)); cnt = np.zeros(len(b)); m = 0
        for x1, y1, x2, y2, s, _ in b:
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += s * np.array([x1, y1, x2, y2]); w[j] += s; cnt[j] += 1
                    fb[j] = acc[j] / w[j]
                    continue
            acc[m] = s * np.array([x1, y1, x2, y2]); w[m] = s; cnt[m] = 1; fb[m] = (x1, y1, x2, y2); m += 1
        score = w[:m] / cnt[:m] * np.minimum(cnt[:m], n_views) / n_views
        out.append(np.c_[fb[:m], score, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    res = np.concatenate(out).astype(np.float32)
    return res[np.argsort(-res[:, 4])][:MAX_DET]

def merge(views, how):
    if how is None:                                         # single pass: as predicted
        return views[0][:MAX_DET]
    if how == "nms":
        return nms(np.concatenate(views), NMS_IOU)
    return wbf(views)

STRATEGIES = {                                              # name: (views, merge)
    "S0_single_pass":   (["full"], None),
    "A1_upscaled_pass": (["up"], None),
    "A2_tta_nms":       (["full", "full_flip", "up", "up_flip"], "nms"),
    "A2_tta_wbf":       (["full", "full_flip", "up", "up_flip"], "wbf"),
    "A3_tiles_nms":     (["full", "tiles"], "nms"),
    "A3_tiles_wbf":     (["full", "tiles"], "wbf"),
    "A2+A3_wbf":        (["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"], "wbf"),
}
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def merge_task(args):
    # one image, one strategy (runs in a worker process)
    iid, views, how = args
    t0 = time.perf_counter()
    b = merge(views, how)
    return iid, b, 1000 * (time.perf_counter() - t0)

def merge_all(cache, views, how):
    # merge the cached passes of every frame; returns detections (COCO style) and merge ms per frame
    tasks = [(iid, [cache[iid][v] for v in views], how) for iid in cache]
    # worker processes only for WBF (pure NumPy). NMS uses PyTorch, which can hang inside forked processes
    # once its thread pool was used in the main process; NMS takes only seconds here anyway.
    if MERGE_WORKERS and how == "wbf":
        with Pool(MERGE_WORKERS) as p:
            res = p.map(merge_task, tasks, chunksize=8)
    else:
        res = [merge_task(t) for t in tasks]
    dets = [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b, _ in res for r in b]
    return dets, float(np.mean([t for _, _, t in res]))


def pmap(f, tasks, chunk):
    # NumPy-only work in MERGE_WORKERS processes (serial with 0)
    if not MERGE_WORKERS:
        return [f(t) for t in tasks]
    with Pool(MERGE_WORKERS) as p:
        return p.map(f, tasks, chunksize=chunk)

def merged(cache, ids):
    # six-pass WBF of each image (strategy A2+A3_wbf), rounded like the submission CSVs (score 4, box 1 decimal)
    tasks = [(iid, [cache[iid][v] for v in ALL_VIEWS], "wbf") for iid in ids]
    res = pmap(merge_task, tasks, 8)
    return {iid: csv_round(b) for iid, b, _ in res}

def csv_round(b):
    # same precision as a submission row: the val numbers then match what the test CSVs contain
    b = b.copy().astype(np.float64)
    wh = np.round(b[:, 2:4] - b[:, 0:2], 1); b[:, 0:2] = np.round(b[:, 0:2], 1); b[:, 2:4] = b[:, 0:2] + wh
    b[:, 4] = np.round(b[:, 4], 4)
    return b.astype(np.float32)

def to_dets(by):
    return [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b in by.items() for r in b]


def read_submission(path):
    # submission CSV -> {image_id: (N, 6) [x1, y1, x2, y2, score, class]}
    out = {}
    for iid, s in pd.read_csv(path).fillna("").itertuples(index=False):
        v = np.array(s.split(), dtype=np.float64).reshape(-1, 6)       # class score x y w h
        out[iid] = np.c_[v[:, 2], v[:, 3], v[:, 2] + v[:, 4], v[:, 3] + v[:, 5], v[:, 1], v[:, 0]].astype(np.float32)
    return out


def fuse(parts, w, iou_thr, p):
    # parts: one (N, 6) array [x1, y1, x2, y2, score, class] per model; w: model weights (members with weight 0 are left out)
    keep = [k for k in range(len(parts)) if w[k] > 0]
    parts, w = [parts[k] for k in keep], np.asarray([w[k] for k in keep], float)
    w = w / w.sum(); M = len(parts)
    allb = np.concatenate([np.c_[a, np.full(len(a), k)] for k, a in enumerate(parts)]) if sum(map(len, parts)) else np.zeros((0, 7))
    out = []
    for c in np.unique(allb[:, 5]):
        x = allb[allb[:, 5] == c]
        x = x[np.argsort(-x[:, 4] * w[x[:, 6].astype(int)], kind="stable")]   # best weighted score first
        n = len(x)
        acc = np.zeros((n, 4)); fb = np.zeros((n, 4)); sw = np.zeros(n); best = np.zeros((n, M)); m = 0
        for x1, y1, x2, y2, s, _, k in x:
            k = int(k); ws = s * w[k]
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                iou[best[:m, k] > 0] = 0                               # at most one box per model in a fused box
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += ws * np.array([x1, y1, x2, y2]); sw[j] += ws; fb[j] = acc[j] / sw[j]
                    best[j, k] = s
                    continue
            acc[m] = ws * np.array([x1, y1, x2, y2]); sw[m] = ws; fb[m] = (x1, y1, x2, y2); best[m] = 0; best[m, k] = s; m += 1
        found = best[:m] > 0
        share = (found * w).sum(1)                                     # weight share of the models that found the box
        mean = (best[:m] * w).sum(1) / share                           # weighted mean score of those models
        out.append(np.c_[fb[:m], mean * share ** p, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    r = np.concatenate(out).astype(np.float32)
    return r[np.argsort(-r[:, 4])][:MAX_DET]

def fuse_task(args):
    iid, parts, w, iou_thr, p = args
    return iid, fuse(parts, w, iou_thr, p)

def fuse_all(outs, ids, w, iou_thr, p):
    # outs: list of {image_id: (N, 6)}, one per member
    empty = np.zeros((0, 6), np.float32)
    return dict(pmap(fuse_task, [(i, [o.get(i, empty) for o in outs], w, iou_thr, p) for i in ids], 16))


VAL, TEST = {}, {}
t0 = time.time()
for k, f in VAL_PASSES.items():
    with gzip.open(find(f), "rb") as fh:
        c = pickle.load(fh)
    VAL[k] = merged(c, VAL_IDS); del c; gc.collect()
for k, f in TEST_CSVS.items():
    TEST[k] = read_submission(find(f))
for k, f in TEST_PASSES.items():
    with gzip.open(find(f), "rb") as fh:
        c = pickle.load(fh)
    TEST[k] = merged(c, TEST_IDS); del c; gc.collect()
    write_submission(to_dets(TEST[k]), WORK / f"submission_13_{k}.csv")      # each new model alone (for the report)
for k in TEST:
    assert set(TEST[k]) == set(TEST_IDS), f"member {k}: test output covers other images than test_images.csv"
NAMES = list(VAL)
print(f"members {NAMES} loaded in {(time.time() - t0) / 60:.1f} min")

saved /kaggle/working/submission_13_L960s1.csv: 3811 rows, 3811 with detections, 1652530 boxes
saved /kaggle/working/submission_13_L1088.csv: 3811 rows, 3811 with detections, 1619491 boxes
members ['04', '08', '10', 'L960', 'M1280', 'M960b', 'L960s1', 'L1088'] loaded in 10.3 min


### Equal-weight fusions: val check, val scores, test submissions

In [6]:
r11 = pd.read_csv(find("results_11_val.csv")).set_index("variant")
OLD6 = ["04", "08", "10", *NEW]
EQ = "fusion " + " ".join(f"{k}=1" for k in OLD6) + " iou=0.55 p=1"
assert EQ in r11.index, f"row {EQ!r} not in results_11_val.csv"
REF_EQ = float(r11.loc[EQ, "mAP"])
IOU, P = 0.55, 1.0                                    # notebook 12's settings, not re-tuned

rows = []
def score(label, by):
    m = coco_eval(VAL_GT, to_dets(by), VAL_IDS)
    rows.append({"variant": label, **m})
    print(f"{label:44s} mAP {m['mAP']:.4f} | AP50 {m['AP50']:.4f} | AP_s {m['AP_s']:.4f}", flush=True)
    return m["mAP"]

def equal(members, ids, outs):
    return fuse_all([outs[k] for k in members], ids, [1.0] * len(members), IOU, P)

for k in NEW13:
    score(f"{k} alone", VAL[k])
eq6 = score("equal, 6 members (notebook 12)", equal(OLD6, VAL_IDS, VAL))
assert abs(eq6 - REF_EQ) < 0.002, f"6-member equal fusion gives {eq6:.4f}, notebook 11/12 had {REF_EQ:.4f}: check the inputs"
ALL8 = OLD6 + NEW13
NO0408 = [k for k in ALL8 if k not in ("04", "08")]
eq8 = score("equal, 8 members", equal(ALL8, VAL_IDS, VAL))
eqx = score("equal, 6 members without 04, 08 (info)", equal(NO0408, VAL_IDS, VAL))
pd.DataFrame(rows).to_csv(WORK / "results_14_val.csv", index=False)

write_submission(to_dets(equal(ALL8, TEST_IDS, TEST)), WORK / "submission_14_equal8.csv")
write_submission(to_dets(equal(NO0408, TEST_IDS, TEST)), WORK / "submission_14_no04_08.csv")
print(f"\nval: 6 members {eq6:.4f} | 8 members {eq8:.4f} ({eq8 - eq6:+.4f}) | without 04, 08 {eqx:.4f}")
print("submit submission_14_equal8.csv (leaderboard so far: 0.2964 with notebook 12's 6-member equal fusion)")

L960s1 alone                                 mAP 0.2303 | AP50 0.4415 | AP_s 0.1395
L1088 alone                                  mAP 0.2389 | AP50 0.4580 | AP_s 0.1608
equal, 6 members (notebook 12)               mAP 0.2701 | AP50 0.5125 | AP_s 0.1572
equal, 8 members                             mAP 0.2721 | AP50 0.5137 | AP_s 0.1651
equal, 6 members without 04, 08 (info)       mAP 0.2737 | AP50 0.5153 | AP_s 0.1624
saved /kaggle/working/submission_14_equal8.csv: 3811 rows, 3811 with detections, 1900863 boxes
saved /kaggle/working/submission_14_no04_08.csv: 3811 rows, 3811 with detections, 1887665 boxes
submit submission_14_equal8.csv (leaderboard so far: 0.2964 with notebook 12's 6-member equal fusion)


**Observations:**
- The second YOLO11l at 960 px (seed 1) reaches only 0.230 on val vs 0.253 for the first: the seed alone changes val mAP by 0.023, so single-model differences of this size are mostly noise. YOLO11l at 1088 px reaches 0.239, with the best AP_small of all models (0.161).
- The two new members add little: 8 members 0.272 vs 0.270 with 6 (equal weights). Without the two weakest members (04, 08) val is 0.274, but choosing members on val is the kind of fitting that did not transfer in notebook 12.
- Leaderboard: `submission_14_equal8.csv` public 0.2972, private 0.2606, **the best of all submissions on both** (notebook 12: 0.2964 / 0.2591). `submission_14_no04_08.csv`, chosen by its higher val, scored lower: 0.2915 / 0.2547. Leaving out members because they are weaker on val did not transfer to test.